In [ ]:
import pandas as pd
dir = '/kaggle/input/gkg-and-stooq-silver-2020-to-2025/news_silver.csv'
df = pd.read_csv(dir)
df['ticker'] = df['ticker'].replace({
    'NVDA': 'Nvidia',
    'MSFT': 'Microsoft'
})

In [ ]:
!git clone https://github.com/kevinscaria/instructabsa.git
%cd instructabsa
!pip install -r requirements.txt

import os
import ast

In [ ]:
# Cell 2: Create Instructions and Data Prep Scripts
# 1. instructions.py
with open("instructions.py", "w") as f:
    f.write('''class InstructionsHandler:
    def __init__(self):
        self.ate = {
            'bos_instruct1': """Definition: The output will be the financial entities or aspects (both implicit and explicit) which have an associated sentiment/opinion extracted from the input text. In cases where there are no aspects the output should be noaspectterm.\\n    Positive example 1-\\n    input: Net profit surged by 20% in the last quarter surpassing analyst estimates.\\n    output: Net profit\\n    Positive example 2-\\n    input: The company's balance sheet remains strong with high liquidity.\\n    output: balance sheet, liquidity\\n    Negative example 1-\\n    input: Shares of ABC Corp plummeted due to the scandal.\\n    output: Shares\\n    Negative example 2-\\n    input: High inflation continues to hurt the operating margin.\\n    output: operating margin\\n    Neutral example 1-\\n    input: The board of directors announced a meeting scheduled for next Monday.\\n    output: board of directors\\n    Neutral example 2-\\n    input: SpiceJet to issue 6.4 crore warrants to promoters.\\n    output: SpiceJet\\n    Now complete the following example-\\n    input: """,
            'bos_instruct2': """Definition: The output will be the financial entities or aspects (both implicit and explicit) which have an associated sentiment/opinion extracted from the input text. In cases where there are no aspects the output should be noaspectterm.\\n    Positive example 1-\\n    input: Net profit surged by 20% in the last quarter surpassing analyst estimates.\\n    output: Net profit\\n    Positive example 2-\\n    input: The company's balance sheet remains strong with high liquidity.\\n    output: balance sheet, liquidity\\n    Negative example 1-\\n    input: Shares of ABC Corp plummeted due to the scandal.\\n    output: Shares\\n    Negative example 2-\\n    input: High inflation continues to hurt the operating margin.\\n    output: operating margin\\n    Neutral example 1-\\n    input: The board of directors announced a meeting scheduled for next Monday.\\n    output: board of directors\\n    Neutral example 2-\\n    input: SpiceJet to issue 6.4 crore warrants to promoters.\\n    output: SpiceJet\\n    Now complete the following example-\\n    input: """,
            'eos_instruct': ' \\noutput:'
        }
        self.atsc = {
            'bos_instruct1': """Definition: The output will be 'positive' if the sentiment of the identified financial entity or aspect in the input is positive (good news, growth, profit). If the sentiment is negative (loss, drop, risk), the answer will be 'negative'. Otherwise, the output should be 'neutral'. For aspects which are classified as noaspectterm, the sentiment is none.\\n    Positive example 1-\\n    input: Profits for Apple surged by 20% this quarter exceeding expectations. The aspect is Profits.\\n    output: positive\\n    Positive example 2-\\n    input: The bank maintains a healthy capital adequacy ratio. The aspect is capital adequacy ratio.\\n    output: positive\\n    Negative example 1-\\n    input: Stocks of Tesla fell sharply due to production delays. The aspect is Stocks.\\n    output: negative\\n    Negative example 2-\\n    input: Rising debt levels are a major concern for the investors. The aspect is debt levels.\\n    output: negative\\n    Neutral example 1-\\n    input: SpiceJet to issue 6.4 crore warrants to promoters. The aspect is SpiceJet.\\n    output: neutral\\n    Neutral example 2-\\n    input: The merger discussion is still ongoing with no final decision. The aspect is merger.\\n    output: neutral\\n    Now complete the following example-\\n    input: """,
            'bos_instruct2': """Definition: The output will be 'positive' if the sentiment of the identified financial entity or aspect in the input is positive (good news, growth, profit). If the sentiment is negative (loss, drop, risk), the answer will be 'negative'. Otherwise, the output should be 'neutral'. For aspects which are classified as noaspectterm, the sentiment is none.\\n    Positive example 1-\\n    input: Profits for Apple surged by 20% this quarter exceeding expectations. The aspect is Profits.\\n    output: positive\\n    Positive example 2-\\n    input: The bank maintains a healthy capital adequacy ratio. The aspect is capital adequacy ratio.\\n    output: positive\\n    Negative example 1-\\n    input: Stocks of Tesla fell sharply due to production delays. The aspect is Stocks.\\n    output: negative\\n    Negative example 2-\\n    input: Rising debt levels are a major concern for the investors. The aspect is debt levels.\\n    output: negative\\n    Neutral example 1-\\n    input: SpiceJet to issue 6.4 crore warrants to promoters. The aspect is SpiceJet.\\n    output: neutral\\n    Neutral example 2-\\n    input: The merger discussion is still ongoing with no final decision. The aspect is merger.\\n    output: neutral\\n    Now complete the following example-\\n    input: """,
            'delim_instruct': ' The aspect is ',
            'eos_instruct': ' \\noutput:'
        }
        self.joint = {
            'bos_instruct1': """Definition: The output will be the financial aspects and their sentiment polarity. Format: aspect:sentiment.\\n    Positive example 1-\\n    input: Revenue grew significantly.\\n    output: Revenue:positive\\n    Negative example 1-\\n    input: Costs are spiraling out of control.\\n    output: Costs:negative\\n    Neutral example 1-\\n    input: The CEO spoke at the conference.\\n    output: CEO:neutral\\n    Now complete the following example-\\n    input: """,
            'bos_instruct2': """Definition: The output will be the financial aspects and their sentiment polarity. Format: aspect:sentiment.\\n    Positive example 1-\\n    input: Revenue grew significantly.\\n    output: Revenue:positive\\n    Negative example 1-\\n    input: Costs are spiraling out of control.\\n    output: Costs:negative\\n    Neutral example 1-\\n    input: The CEO spoke at the conference.\\n    output: CEO:neutral\\n    Now complete the following example-\\n    input: """,
            'eos_instruct': ' \\noutput:'
        }

    def load_instruction_set1(self):
        self.ate = self.ate
        self.atsc = self.atsc
        self.joint = self.joint

    def load_instruction_set2(self):
        self.ate = self.ate
        self.atsc = self.atsc
        self.joint = self.joint
''')

# 2. InstructABSA/data_prep.py
with open("InstructABSA/data_prep.py", "w") as f:
    f.write('''from datasets import Dataset
from datasets.dataset_dict import DatasetDict
import ast

class DatasetLoader:
    def __init__(self, train_df_id=None, test_df_id=None,
                 train_df_ood=None, test_df_ood=None, sample_size=1,
                 val_df_id=None, val_df_ood=None):

        self.train_df_id = train_df_id.sample(frac=sample_size, random_state=1999) if train_df_id is not None else train_df_id
        self.test_df_id = test_df_id
        self.train_df_ood = train_df_ood
        self.test_df_ood = test_df_ood
        self.val_df_id = val_df_id
        self.val_df_ood = val_df_ood

    def reconstruct_strings(self, df, col):
        reconstructed_col = []
        for text in df[col]:
            try:
                if isinstance(text, (list, dict)):
                    reconstructed_col.append(text)
                elif isinstance(text, str):
                    if text == '[]':
                        reconstructed_col.append([])
                    else:
                        reconstructed_col.append(ast.literal_eval(text))
                else:
                    reconstructed_col.append([])
            except (ValueError, SyntaxError):
                reconstructed_col.append([])
        df[col] = reconstructed_col
        return df

    def extract_rowwise_aspect_polarity(self, df, on, key, min_val = None):
        try:
            df.iloc[0][on][0][key]
        except:
            df = self.reconstruct_strings(df, on)

        df['len'] = df[on].apply(lambda x: len(x))
        if min_val is not None:
            df.loc[df['len'] == 0, 'len'] = min_val
        df = df.loc[df.index.repeat(df['len'])]
        df['record_idx'] = df.groupby(df.index).cumcount()
        df['aspect'] = df[[on, 'record_idx']].apply(lambda x : (x[0][x[1]][key], x[0][x[1]]['polarity']) if len(x[0]) != 0 else ('',''), axis=1)
        df['polarity'] = df['aspect'].apply(lambda x: x[-1])
        df['aspect'] = df['aspect'].apply(lambda x: x[0])
        df = df.drop(['len', 'record_idx'], axis=1).reset_index(drop = True)
        return df

    def extract_rowwise_aspect_opinions(self, df, aspect_col, opinion_col, key, min_val = None):
        df['len'] = df[aspect_col].apply(lambda x: len(x))
        if min_val is not None:
            df.loc[df['len'] == 0, 'len'] = min_val
        df = df.loc[df.index.repeat(df['len'])]
        df['record_idx'] = df.groupby(df.index).cumcount()
        df['aspect'] = df[[aspect_col, 'record_idx']].apply(lambda x : x[0][x[1]][key] if len(x[0]) != 0 else '', axis=1)
        df['opinion_term'] = df[[opinion_col, 'record_idx']].apply(lambda x : x[0][x[1]][key] if len(x[0]) != 0 else '', axis=1)
        df['aspect'] = df['aspect'].apply(lambda x: ' '.join(x))
        df['opinion_term'] = df['opinion_term'].apply(lambda x: ' '.join(x))
        df = df.drop(['len', 'record_idx'], axis=1).reset_index(drop = True)
        return df

    def create_data_in_ate_format(self, df, key, text_col, aspect_col, bos_instruction = '', eos_instruction = ''):
        if df is None: return
        try: df.iloc[0][aspect_col][0][key]
        except: df = self.reconstruct_strings(df, aspect_col)
        df['labels'] = df[aspect_col].apply(lambda x: ', '.join([i[key] for i in x]))
        df['text'] = df[text_col].apply(lambda x: bos_instruction + x + eos_instruction)
        return df

    def create_data_in_atsc_format(self, df, on, key, text_col, aspect_col, bos_instruction = '', delim_instruction = '', eos_instruction = ''):
        if df is None: return
        df = self.extract_rowwise_aspect_polarity(df, on=on, key=key, min_val=1)
        df['text'] = df[[text_col, aspect_col]].apply(lambda x: bos_instruction + x[0] + delim_instruction + x[1] + eos_instruction, axis=1)
        df = df.rename(columns = {'polarity': 'labels'})
        return df

    def create_data_in_aspe_format(self, df, key, label_key, text_col, aspect_col, bos_instruction = '', eos_instruction = ''):
        if df is None: return
        try: df.iloc[0][aspect_col][0][key]
        except: df = self.reconstruct_strings(df, aspect_col)
        df['labels'] = df[aspect_col].apply(lambda x: ', '.join([f"{i[key]}:{i[label_key]}" for i in x]))
        df['text'] = df[text_col].apply(lambda x: bos_instruction + x + eos_instruction)
        return df

    def create_data_in_aooe_format(self, df, aspect_col, opinion_col, key, text_col, bos_instruction = '', delim_instruction = '', eos_instruction = ''):
        if df is None: return
        df = self.extract_rowwise_aspect_opinions(df, aspect_col=aspect_col, opinion_col=opinion_col, key=key, min_val=1)
        df['text'] = df[[text_col, 'aspect']].apply(lambda x: bos_instruction + x[0] + delim_instruction + x[1] + eos_instruction, axis=1)
        df = df.rename(columns = {'opinion_term': 'labels'})
        return df

    def create_data_in_aope_format(self, df, key, text_col, aspect_col, opinion_col, bos_instruction = '', eos_instruction = ''):
        df['labels'] = df[[aspect_col, opinion_col]].apply(lambda x: ', '.join([f"{' '.join(i[key])}:{' '.join(j[key])}" for i, j in zip(x[0], x[1])]), axis=1)
        df['text'] = df[text_col].apply(lambda x: bos_instruction + x + eos_instruction)
        return df

    def create_data_in_aoste_format(self, df, key, label_key, text_col, aspect_col, opinion_col, bos_instruction = '', eos_instruction = ''):
        label_map = {'POS':'positive', 'NEG':'negative', 'NEU':'neutral'}
        df['labels'] = df[[aspect_col, opinion_col]].apply(lambda x: ', '.join([f"{' '.join(i[key])}:{' '.join(j[key])}:{label_map[i[label_key]]}" for i, j in zip(x[0], x[1])]), axis=1)
        df['text'] = df[text_col].apply(lambda x: bos_instruction + x + eos_instruction)
        return df

    def set_data_for_training_semeval(self, tokenize_function):
        dataset_dict_id, dataset_dict_ood = {}, {}
        if self.train_df_id is not None: dataset_dict_id['train'] = Dataset.from_pandas(self.train_df_id)
        if self.test_df_id is not None: dataset_dict_id['test'] = Dataset.from_pandas(self.test_df_id)
        if self.val_df_id is not None: dataset_dict_id['validation'] = Dataset.from_pandas(self.val_df_id)

        if len(dataset_dict_id) > 1:
            indomain_dataset = DatasetDict(dataset_dict_id)
            indomain_tokenized_datasets = indomain_dataset.map(tokenize_function, batched=True)
        else:
            indomain_dataset, indomain_tokenized_datasets = {}, {}

        if self.train_df_ood is not None: dataset_dict_ood['train'] = Dataset.from_pandas(self.train_df_ood)
        if self.test_df_ood is not None: dataset_dict_ood['test'] = Dataset.from_pandas(self.test_df_ood)
        if self.val_df_ood is not None: dataset_dict_ood['validation'] = Dataset.from_pandas(self.val_df_ood)

        if len(dataset_dict_id) > 1:
            other_domain_dataset = DatasetDict(dataset_dict_ood)
            other_domain_tokenized_dataset = other_domain_dataset.map(tokenize_function, batched=True)
        else:
            other_domain_dataset, other_domain_tokenized_dataset = {}, {}

        return indomain_dataset, indomain_tokenized_datasets, other_domain_dataset, other_domain_tokenized_dataset
''')

# 3. InstructABSA/utils.py (Initial version for training)
with open("InstructABSA/utils.py", "w") as f:
    f.write('''import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
from transformers import (
    DataCollatorForSeq2Seq, AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Trainer, Seq2SeqTrainer
)

class T5Generator:
    def __init__(self, model_checkpoint):
        self.tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
        self.data_collator = DataCollatorForSeq2Seq(self.tokenizer)
        self.device = 'cuda' if torch.has_cuda else ('mps' if torch.has_mps else 'cpu')

    def tokenize_function_inputs(self, sample):
        model_inputs = self.tokenizer(sample['text'], max_length=512, truncation=True)
        labels = self.tokenizer(sample["labels"], max_length=64, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    def train(self, tokenized_datasets, **kwargs):
        args = Seq2SeqTrainingArguments(**kwargs)
        eval_ds = tokenized_datasets.get("validation")
        if eval_ds is None: eval_ds = tokenized_datasets.get("test")
        trainer = Seq2SeqTrainer(
            self.model, args, train_dataset=tokenized_datasets["train"],
            eval_dataset=eval_ds, tokenizer=self.tokenizer, data_collator=self.data_collator,
        )
        torch.cuda.empty_cache()
        trainer.train()
        trainer.save_model()
        return trainer

    def get_labels(self, tokenized_dataset, batch_size = 4, max_length = 128, sample_set = 'train'):
        def collate_fn(batch):
            input_ids = [torch.tensor(example['input_ids']) for example in batch]
            input_ids = pad_sequence(input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id)
            return input_ids
        dataloader = DataLoader(tokenized_dataset[sample_set], batch_size=batch_size, collate_fn=collate_fn)
        predicted_output = []
        self.model.to(self.device)
        for batch in tqdm(dataloader):
            batch = batch.to(self.device)
            output_ids = self.model.generate(batch, max_length = max_length)
            output_texts = self.tokenizer.batch_decode(output_ids, skip_special_tokens=True)
            for output_text in output_texts: predicted_output.append(output_text)
        return predicted_output

    def get_metrics(self, y_true, y_pred, is_triplet_extraction=False):
        # Simplified for brevity in this combined script
        return precision_score(y_true, y_pred, average='macro'), recall_score(y_true, y_pred, average='macro'), f1_score(y_true, y_pred, average='macro'), None

class T5Classifier:
    def __init__(self, model_checkpoint):
        self.tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, force_download = True)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint, force_download = True)
        self.data_collator = DataCollatorForSeq2Seq(self.tokenizer)
        self.device = 'cuda' if torch.has_cuda else ('mps' if torch.has_mps else 'cpu')

    def tokenize_function_inputs(self, sample):
        sample['input_ids'] = self.tokenizer(sample["text"], max_length = 512, truncation = True).input_ids
        sample['labels'] = self.tokenizer(sample["labels"], max_length = 64, truncation = True).input_ids
        return sample

    def train(self, tokenized_datasets, **kwargs):
        args = Seq2SeqTrainingArguments(**kwargs)
        eval_ds = tokenized_datasets.get("validation")
        if eval_ds is None: eval_ds = tokenized_datasets.get("test")
        trainer = Trainer(
            self.model, args, train_dataset=tokenized_datasets["train"],
            eval_dataset=eval_ds, tokenizer=self.tokenizer, data_collator = self.data_collator
        )
        torch.cuda.empty_cache()
        trainer.train()
        trainer.save_model()
        return trainer

    def get_labels(self, tokenized_dataset, batch_size = 4, sample_set = 'train'):
        def collate_fn(batch):
            input_ids = [torch.tensor(example['input_ids']) for example in batch]
            input_ids = pad_sequence(input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id)
            return input_ids
        dataloader = DataLoader(tokenized_dataset[sample_set], batch_size=batch_size, collate_fn=collate_fn)
        predicted_output = []
        self.model.to(self.device)
        for batch in tqdm(dataloader):
            batch = batch.to(self.device)
            output_ids = self.model.generate(batch)
            output_texts = self.tokenizer.batch_decode(output_ids, skip_special_tokens=True)
            for output_text in output_texts: predicted_output.append(output_text)
        return predicted_output

    def get_metrics(self, y_true, y_pred):
        return precision_score(y_true, y_pred, average='macro'), recall_score(y_true, y_pred, average='macro'), f1_score(y_true, y_pred, average='macro'), accuracy_score(y_true, y_pred)
''')

# 4. run_model.py
with open("run_model.py", "w") as f:
    f.write('''import os
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import torch
from InstructABSA.data_prep import DatasetLoader
from InstructABSA.utils import T5Generator, T5Classifier
from InstructABSA.config import Config
from instructions import InstructionsHandler

try: use_mps = True if torch.has_mps else False
except: use_mps = False

config = Config()
instruct_handler = InstructionsHandler()
if config.inst_type == 1: instruct_handler.load_instruction_set1()
else: instruct_handler.load_instruction_set2()

if config.mode == 'train' and config.id_tr_data_path is None: raise Exception('Provide training data path.')
if config.mode == 'eval' and config.id_te_data_path is None and config.ood_te_data_path is None: raise Exception('Provide testing data path.')

if config.experiment_name is not None and config.mode == 'train':
    model_checkpoint = config.model_checkpoint
    model_out_path = os.path.join(config.output_dir, config.task, f"{model_checkpoint.replace('/', '')}-{config.experiment_name}")
else:
    model_checkpoint = config.model_checkpoint
    model_out_path = config.model_checkpoint

id_tr_data_path = config.id_tr_data_path
ood_tr_data_path = config.ood_tr_data_path
id_te_data_path = config.id_te_data_path
ood_te_data_path = config.ood_te_data_path

if config.mode != 'cli':
    id_tr_df, id_te_df, ood_tr_df, ood_te_df = None, None, None, None
    if id_tr_data_path: id_tr_df = pd.read_csv(id_tr_data_path)
    if id_te_data_path: id_te_df = pd.read_csv(id_te_data_path)
    if ood_tr_data_path: ood_tr_df = pd.read_csv(ood_tr_data_path)
    if ood_te_data_path: ood_te_df = pd.read_csv(ood_te_data_path)
    print('Loaded data...')

training_args = {
    'output_dir': model_out_path,
    'eval_strategy': config.evaluation_strategy if config.id_te_data_path is not None else 'no',
    'learning_rate': config.learning_rate,
    'per_device_train_batch_size': config.per_device_train_batch_size,
    'per_device_eval_batch_size': config.per_device_eval_batch_size,
    'num_train_epochs': config.num_train_epochs,
    'weight_decay': config.weight_decay,
    'warmup_ratio': config.warmup_ratio,
    'save_strategy': config.save_strategy,
    'load_best_model_at_end': config.load_best_model_at_end,
    'push_to_hub': config.push_to_hub,
    'eval_accumulation_steps': config.eval_accumulation_steps,
    'predict_with_generate': config.predict_with_generate,
    'use_mps_device': use_mps
}

if config.set_instruction_key == 1:
    indomain, outdomain = 'bos_instruct1', 'bos_instruct2'
else:
    indomain, outdomain = 'bos_instruct2', 'bos_instruct1'

if config.task == 'ate':
    t5_exp = T5Generator(model_checkpoint)
    bos_instruction_id = instruct_handler.ate[indomain]
    bos_instruction_ood = instruct_handler.ate[outdomain] if (ood_tr_data_path or ood_te_data_path) else ''
    eos_instruction = instruct_handler.ate['eos_instruct']
elif config.task == 'atsc':
    t5_exp = T5Classifier(model_checkpoint)
    bos_instruction_id = instruct_handler.atsc[indomain]
    bos_instruction_ood = instruct_handler.atsc[outdomain] if (ood_tr_data_path or ood_te_data_path) else ''
    delim_instruction = instruct_handler.atsc['delim_instruct']
    eos_instruction = instruct_handler.atsc['eos_instruct']
elif config.task == 'joint':
    t5_exp = T5Generator(model_checkpoint)
    bos_instruction_id = instruct_handler.joint[indomain]
    bos_instruction_ood = instruct_handler.joint[outdomain] if (ood_tr_data_path or ood_te_data_path) else ''
    eos_instruction = instruct_handler.joint['eos_instruct']

if config.mode != 'cli':
    loader = DatasetLoader(id_tr_df, id_te_df, ood_tr_df, ood_te_df, config.sample_size)

    if config.task == 'ate':
        if loader.train_df_id is not None: loader.train_df_id = loader.create_data_in_ate_format(loader.train_df_id, 'term', 'raw_text', 'aspectTerms', bos_instruction_id, eos_instruction)
        if loader.test_df_id is not None: loader.test_df_id = loader.create_data_in_ate_format(loader.test_df_id, 'term', 'raw_text', 'aspectTerms', bos_instruction_id, eos_instruction)
    elif config.task == 'atsc':
        if loader.train_df_id is not None: loader.train_df_id = loader.create_data_in_atsc_format(loader.train_df_id, 'aspectTerms', 'term', 'raw_text', 'aspect', bos_instruction_id, delim_instruction, eos_instruction)
        if loader.test_df_id is not None: loader.test_df_id = loader.create_data_in_atsc_format(loader.test_df_id, 'aspectTerms', 'term', 'raw_text', 'aspect', bos_instruction_id, delim_instruction, eos_instruction)

    id_ds, id_tokenized_ds, ood_ds, ood_tokenized_ds = loader.set_data_for_training_semeval(t5_exp.tokenize_function_inputs)

    if config.mode == 'train':
        model_trainer = t5_exp.train(id_tokenized_ds, **training_args)
        print('Model saved at: ', model_out_path)
    elif config.mode == 'eval':
        print('Model loaded from: ', model_checkpoint)
        if id_tokenized_ds.get("test") is not None:
            id_te_pred_labels = t5_exp.get_labels(tokenized_dataset = id_tokenized_ds, sample_set = 'test', batch_size=config.per_device_eval_batch_size, max_length = config.max_token_length)
            id_te_df = pd.DataFrame(id_ds['test'])[['text', 'labels']]
            id_te_df['pred_labels'] = id_te_pred_labels
            id_te_df.to_csv(os.path.join(config.output_path, f'{config.experiment_name}_id_test.csv'), index=False)
            print('*****Test Metrics*****')
            precision, recall, f1, accuracy = t5_exp.get_metrics(id_te_df['labels'], id_te_pred_labels)
            print('Precision: ', precision)
            print('Recall: ', recall)
            print('F1-Score: ', f1)
            if config.task == 'atsc': print('Accuracy: ', accuracy)
else:
    print("CLI mode not fully implemented in this unified script.")
''')

print("Configuration files created successfully.")

In [ ]:
# Cell 5: Apply Inference Patches (Fixing utils.py and run_model.py)

# 1. Update run_model.py to handle evaluation_strategy naming if needed
# (Actually, the version written in Cell 2 already handles this via the if-check,
# but we will ensure the variable naming aligns with what Config expects).
# We primarily need to update utils.py for the max_new_tokens fix.

updated_utils_code = """
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
from transformers import (
    DataCollatorForSeq2Seq, AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Trainer, Seq2SeqTrainer
)

class T5Generator:
    def __init__(self, model_checkpoint):
        self.tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
        self.data_collator = DataCollatorForSeq2Seq(self.tokenizer)
        self.device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')

    def tokenize_function_inputs(self, sample):
        model_inputs = self.tokenizer(sample['text'], max_length=512, truncation=True)
        labels = self.tokenizer(sample["labels"], max_length=64, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    def train(self, tokenized_datasets, **kwargs):
        args = Seq2SeqTrainingArguments(**kwargs)
        eval_ds = tokenized_datasets.get("validation")
        if eval_ds is None: eval_ds = tokenized_datasets.get("test")
        trainer = Seq2SeqTrainer(
            self.model, args, train_dataset=tokenized_datasets["train"],
            eval_dataset=eval_ds, tokenizer=self.tokenizer, data_collator=self.data_collator,
        )
        torch.cuda.empty_cache()
        trainer.train()
        trainer.save_model()
        return trainer

    def get_labels(self, tokenized_dataset, batch_size=4, max_length=128, sample_set='train'):
        def collate_fn(batch):
            input_ids = [torch.tensor(example['input_ids']) for example in batch]
            input_ids = pad_sequence(input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id)
            return input_ids
        dataloader = DataLoader(tokenized_dataset[sample_set], batch_size=batch_size, collate_fn=collate_fn)
        predicted_output = []
        self.model.to(self.device)
        for batch in tqdm(dataloader):
            batch = batch.to(self.device)
            # FIXED: max_new_tokens
            output_ids = self.model.generate(batch, max_new_tokens=max_length)
            output_texts = self.tokenizer.batch_decode(output_ids, skip_special_tokens=True)
            for output_text in output_texts: predicted_output.append(output_text)
        return predicted_output

    def get_metrics(self, y_true, y_pred, is_triplet_extraction=False):
        return precision_score(y_true, y_pred, average='macro'), recall_score(y_true, y_pred, average='macro'), f1_score(y_true, y_pred, average='macro'), accuracy_score(y_true, y_pred)

class T5Classifier:
    def __init__(self, model_checkpoint):
        self.tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, force_download=True)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint, force_download=True)
        self.data_collator = DataCollatorForSeq2Seq(self.tokenizer)
        self.device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')

    def tokenize_function_inputs(self, sample):
        sample['input_ids'] = self.tokenizer(sample["text"], max_length=512, truncation=True).input_ids
        sample['labels'] = self.tokenizer(sample["labels"], max_length=64, truncation=True).input_ids
        return sample

    def train(self, tokenized_datasets, **kwargs):
        args = Seq2SeqTrainingArguments(**kwargs)
        eval_ds = tokenized_datasets.get("validation")
        if eval_ds is None: eval_ds = tokenized_datasets.get("test")
        trainer = Trainer(
            self.model, args, train_dataset=tokenized_datasets["train"],
            eval_dataset=eval_ds, tokenizer=self.tokenizer, data_collator=self.data_collator
        )
        torch.cuda.empty_cache()
        trainer.train()
        trainer.save_model()
        return trainer

    def get_labels(self, tokenized_dataset, batch_size=4, max_length=128, sample_set='train'):
        def collate_fn(batch):
            input_ids = [torch.tensor(example['input_ids']) for example in batch]
            input_ids = pad_sequence(input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id)
            return input_ids
        dataloader = DataLoader(tokenized_dataset[sample_set], batch_size=batch_size, collate_fn=collate_fn)
        predicted_output = []
        self.model.to(self.device)
        for batch in tqdm(dataloader):
            batch = batch.to(self.device)
            # FIXED: max_new_tokens (This was the critical bug in the inference NB)
            output_ids = self.model.generate(batch, max_new_tokens=max_length)
            output_texts = self.tokenizer.batch_decode(output_ids, skip_special_tokens=True)
            for output_text in output_texts: predicted_output.append(output_text)
        return predicted_output

    def get_metrics(self, y_true, y_pred):
        return precision_score(y_true, y_pred, average='macro'), recall_score(y_true, y_pred, average='macro'), \
            f1_score(y_true, y_pred, average='macro'), accuracy_score(y_true, y_pred)
"""

with open("InstructABSA/utils.py", "w") as f:
    f.write(updated_utils_code)

print("Inference patches applied (switched to max_new_tokens for generation).")

In [ ]:
# Cell 8: Optimized Batch Inference for Regression Scores
import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

# --- Configuration ---
MODEL_PATH = "/kaggle/input/finstructabsa/pytorch/default/1/sentfin_model_output/atsc/googleflan-t5-base-run1"
BATCH_SIZE = 64  # Increased for speed on T4 GPU
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- 1. Load Model & Tokenizer ---
print(f"Loading model from {MODEL_PATH} on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# Load in float16 for 2x speedup on T4 GPU
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH, torch_dtype=torch.float16).to(DEVICE)
model.eval()

# --- 2. Pre-calculate Token IDs for Regression ---
# We need the IDs for "positive", "negative", and "neutral" to look up their probabilities
pos_token_id = tokenizer.encode("positive", add_special_tokens=False)[0]
neg_token_id = tokenizer.encode("negative", add_special_tokens=False)[0]
neu_token_id = tokenizer.encode("neutral", add_special_tokens=False)[0]
print(f"Token IDs - Pos: {pos_token_id}, Neg: {neg_token_id}, Neu: {neu_token_id}")

# --- 3. Define Prompt Format ---
def create_prompt(text, aspect):
    definition = (
        "Definition: Determine the sentiment polarity expressed toward the specified aspect. "
        "Output 'positive', 'negative', or 'neutral'.\n"
        "Positive example 1- input: Nvidia–Intel deal was cleared. Aspect: Nvidia. output: positive\n"
        "Negative example 1- input: Nvidia shares slid 5%. Aspect: Nvidia. output: negative\n"
        "Neutral example 1- input: Satya Nadella attended the conference. Aspect: Microsoft. output: neutral\n"
        "Now complete the following example-\n"
    )
    return f"{definition}input: {text} The aspect is {aspect}.\noutput:"

# --- 4. Custom Dataset for Batching ---
class SentimentDataset(Dataset):
    def __init__(self, df):
        self.texts = df['extracted_title'].tolist()
        self.tickers = df['ticker'].tolist()
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        return create_prompt(self.texts[idx], self.tickers[idx])

# --- 5. Batch Inference Function (Regression Logic) ---
def get_sentiment_scores(df, batch_size=BATCH_SIZE):
    dataset = SentimentDataset(df)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    all_scores = []
    
    print(f"Starting inference on {len(df)} rows...")
    with torch.no_grad():
        for batch_prompts in tqdm(loader, desc="Inferencing"):
            # Tokenize batch
            inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(DEVICE)
            
            # Prepare decoder input (start token) for the forward pass
            # T5 needs to know what the "previous" token was to predict the next. 
            # We give it the <pad> token to ask for the *first* generated word.
            decoder_input_ids = torch.full(
                (inputs.input_ids.shape[0], 1), 
                model.config.decoder_start_token_id, 
                device=DEVICE
            )
            
            # Forward pass (Get Logits) - FAST!
            outputs = model(input_ids=inputs.input_ids, decoder_input_ids=decoder_input_ids)
            
            # Extract logits for the first token position [Batch, SeqLen, Vocab] -> [Batch, Vocab]
            next_token_logits = outputs.logits[:, 0, :]
            
            # Select only the logits for our target words
            target_logits = next_token_logits[:, [neg_token_id, neu_token_id, pos_token_id]]
            
            # Softmax to get probabilities that sum to 1
            probs = torch.softmax(target_logits, dim=-1) # shape: [Batch, 3]
            
            # Calculate Regression Score: P(Positive) - P(Negative)
            # Index 0=Neg, 1=Neu, 2=Pos
            scores = probs[:, 2] - probs[:, 0]
            
            # Move to CPU and store
            all_scores.extend(scores.cpu().float().numpy())
            
    return all_scores

# --- 6. Execution ---
# Run on the full dataframe
scores = get_sentiment_scores(df)

# Assign to dataframe
df['sentiment_score'] = scores

# Save
output_filename = 'news_silver_with_scores.csv'
df.to_csv(output_filename, index=False)
print(f"Done! Saved to {output_filename}")
print(df[['extracted_title', 'ticker', 'sentiment_score']].head())

In [ ]:
# Cell 9: Advanced Daily Signal Aggregation

# 1. Ensure date format
df['date'] = pd.to_datetime(df['date'])

# 2. Define custom aggregations
def signal_strength(x):
    # Sum of absolute values implies "Total Information Flow"
    return x.abs().sum()

def polarity_ratio(x):
    # Ratio of bullishness vs total emotional volume (0 to 1 scale)
    pos = x[x > 0].sum()
    neg = x[x < 0].abs().sum()
    total = pos + neg
    if total == 0: return 0.5 # Neutral if no strong sentiment
    return pos / total

# 3. Aggregate by Date and Ticker
daily_signals = df.groupby(['date', 'ticker']).agg(
    sentiment_sum=('sentiment_score', 'sum'),    # Impact (Volume * Polarity)
    sentiment_avg=('sentiment_score', 'mean'),   # Consensus (diluted by neutrals)
    news_count=('extracted_title', 'count'),     # Raw Attention
    polarity_ratio=('sentiment_score', polarity_ratio) # Pure Direction (ignores neutrals)
).reset_index()

# 4. Save
daily_signals = daily_signals.sort_values('date')
daily_signals.to_csv('daily_signals_advanced.csv', index=False)

# 5. Inspection
print(daily_signals.head())

# 6. Visualization: Compare "Avg" vs "Sum"
import matplotlib.pyplot as plt

ticker = 'Nvidia'
data = daily_signals[daily_signals['ticker'] == ticker]

fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.set_xlabel('Date')
ax1.set_ylabel('Sentiment Sum (Impact)', color='tab:blue')
ax1.plot(data['date'], data['sentiment_sum'], color='tab:blue', label='Sentiment Sum')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()  
ax2.set_ylabel('Sentiment Avg (Diluted)', color='tab:orange')  
ax2.plot(data['date'], data['sentiment_avg'], color='tab:orange', linestyle='--', label='Sentiment Avg')
ax2.tick_params(axis='y', labelcolor='tab:orange')

plt.title(f"{ticker}: Impact (Sum) vs Consensus (Avg)")
fig.tight_layout() 
plt.show()